# Building a pharmacophore filter and measuring enrichment

Turn the discriminating feature into a hard library filter, then into a ranking score, and measure EF1%, EF0.5%, BEDROC (α = 20), ROC-AUC and PR-AUC on the labelled DUD-E EGFR subset. A second ranking uses Gobbi Pharm2D similarity to a held-out query set. Ties in every ranking are broken randomly with seed 0 so the actives-first file order cannot inflate early enrichment.

In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from pharmprof.data import load_egfr
from pharmprof.patterns import builtin_patterns
from pharmprof.profiler import profile
from pharmprof.screen import apply_filter, pharm2d_similarity
from pharmprof.metrics import screening_metrics
from pharmprof.io import MolRecord

data = load_egfr(subset=True)
recs = [MolRecord(str(r.mol_id), r.smiles, Chem.MolFromSmiles(r.smiles), int(r.label)) for r in data.itertuples()]
patterns = builtin_patterns("basefeatures")
table = profile(recs, patterns, mode="count", n_jobs=4, progress=False)
table["label"] = [r.label for r in recs]
basic = "basefeatures.PosIonizable.BasicGroup"
print("BasicGroup hit rate  actives", float((table.loc[table.label==1, basic] > 0).mean()),
      "decoys", float((table.loc[table.label==0, basic] > 0).mean()))

/workspace/venvs/pharmprof/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BasicGroup hit rate  actives 0.48523985239852396 decoys 0.0512


### Hard filter: require a basic group

In [2]:
mask = apply_filter(table, require=["BasicGroup"], patterns=patterns)
kept = table.loc[mask]
n, n_act = len(table), int(table.label.sum())
n_keep, n_act_keep = len(kept), int(kept.label.sum())
hr_a = n_act_keep / n_act
hr_i = (n_keep - n_act_keep) / (n - n_act)
print(f"kept {n_keep} / {n}")
print(f"actives {n_act_keep} / {n_act} ({hr_a:.3f})")
print(f"decoys  {n_keep - n_act_keep} / {n - n_act} ({hr_i:.3f})")
print(f"enrichment {hr_a / hr_i:.3f}")

kept 391 / 3042
actives 263 / 542 (0.485)
decoys  128 / 2500 (0.051)
enrichment 9.477


### Ranking by BasicGroup count

The score is simply the match count for that one pattern. Prevalence is 542 / 3,042 = 0.178, so the maximum possible EF1% is 1 / 0.178 ≈ 5.61.

In [3]:
metrics = screening_metrics(table["label"], table[basic].fillna(0))
metrics

n               3042.000000
n_actives        542.000000
roc_auc            0.719285
pr_auc             0.446568
bedroc             0.692170
bedroc_alpha      20.000000
ef0.5              5.238376
ef1                5.425461
dtype: float64

### Gobbi Pharm2D similarity, held-out queries

80 randomly chosen actives (seed 0) are the query set. The test set is the remaining 462 actives plus all 2,500 decoys. Each test molecule is scored by its maximum Gobbi Tanimoto to the query set; a molecule is never compared to itself. **A caveat:** DUD-E decoys were chosen to be topologically dissimilar to the actives, so a 2-D pharmacophore fingerprint is expected to do well on this benchmark.

In [4]:
actives = [r for r in recs if r.label == 1]
decoys = [r for r in recs if r.label == 0]
rng = np.random.default_rng(0)
order = rng.permutation(len(actives))
query = [actives[i] for i in order[:80]]
held = [actives[i] for i in order[80:]]
library = held + decoys
print(f"queries {len(query)}; test actives {len(held)} + decoys {len(decoys)}")

sims = pharm2d_similarity(query, library, aggregation="max", progress=True, exclude_self=True)
score_map = dict(zip(sims.mol_id.astype(str), sims.pharm2d_tanimoto))
y = np.array([r.label for r in library])
s = np.array([score_map[str(r.mol_id)] for r in library], dtype=float)
gobbi = screening_metrics(y, s)
gobbi

queries 80; test actives 462 + decoys 2500


pharm2d: 100%|██████████| 2962/2962 [01:35<00:00, 31.04it/s]


n               2962.000000
n_actives        462.000000
roc_auc            0.962263
pr_auc             0.911606
bedroc             0.968035
bedroc_alpha      20.000000
ef0.5              6.411255
ef1                6.411255
dtype: float64